# 01: Topic inventory

**Goal:** find out what is actually inside the downloaded bags, before writing any loader code.

Specifically:

1. Which topic carries the Xsens GNSS, and at what rate?
2. Which topic carries the OBD forward speed?
3. Which topic carries the ground-truth reference?
4. Which message types are custom to NavINST, and therefore need registering before they can be decoded?
5. Which topics look useful but must **not** be used, and why?

Nothing here is production code. The loader proper goes in `src/data_loader.py` once these answers are known.

> **Sequence:** `Urban04` Day. The indoor garage sequences were abandoned because they ship no GNSS data at all. See `CLAUDE.md` §4.

## What a ROS bag actually is

A **bag** is a recording file. Inside it is a long, time-ordered sequence of **messages**. Each message carries:

- a **topic**, a named channel such as `/xsens/imu`. One sensor's output stream.
- a **message type**, which is a schema, essentially a struct definition, such as `sensor_msgs/msg/Imu`. It says how to interpret the packed bytes.
- a **timestamp**. In fact two of them, which the next cells make visible.

A useful mental image: the bag is a long list of envelopes. Each envelope holds raw bytes plus a number saying which **connection** it belongs to, and a table at the front of the file says what each connection means.

### Why we do not replay the bag

ROS normally *replays* a bag in real time, re-publishing the messages at original speed so that live software cannot tell the difference between the recording and the sensor. That exists for testing a robot's running stack.

It is the wrong tool here, for three reasons:

1. **Speed.** Replaying `Urban04` takes 17 minutes 45 seconds, every single time. Reading it into tables takes about 7 seconds, and filter tuning means iterating dozens of times.
2. **Determinism.** Real-time replay brings scheduling jitter and dropped messages. Offline estimation should give identical numbers on every run, so that a change in the output means a change in the code.
3. **It is the wrong shape.** Replay gives you one message at a time, forwards only. This project constantly wants things like "every GNSS fix between t=300 and t=360", or "join the IMU stream to the reference". Those are table operations, not stream operations.

Replay matters when latency and message ordering are part of what you are testing. This is offline estimation, so a table is the honest representation.

In [1]:
from pathlib import Path

import pandas as pd
from rosbags.highlevel import AnyReader

# Work out the repository root whether this runs from the repo root or from notebooks/.
REPO = Path.cwd()
if not (REPO / "data").exists():
    REPO = REPO.parent

SEQUENCE = REPO / "data" / "raw" / "Urban04" / "2023-11-01-12-16-16-Day"
BAGS = {name: SEQUENCE / f"{name}.bag" for name in ("gnss", "imus", "reference")}

print(REPO)
for name, bag in BAGS.items():
    size = f"{bag.stat().st_size / 1e6:.1f} MB" if bag.exists() else "MISSING"
    print(f"{name + '.bag':16s} {size}")

/Users/xinyue/repos/imu-gnss-localization
gnss.bag         61.9 MB
imus.bag         351.6 MB
reference.bag    10.4 MB


## The reader

`AnyReader` is the format-agnostic entry point in `rosbags`. It works out whether it is looking at a ROS1 `.bag` or a ROS2 directory and handles either. It takes a **list** of paths, because a recording split across several files can be read as one logical stream. That is useful here, since NavINST splits each run by sensor group.

It is a **context manager**: `with AnyReader([...]) as reader:` opens the file and guarantees it is closed again, whether the block finishes normally or raises. Everything you need must be read inside that block, because afterwards the file is closed and the reader is spent.

The attributes that matter:

| Attribute | Meaning |
|---|---|
| `reader.connections` | the list of connections, described below |
| `reader.start_time`, `reader.end_time` | bag extent, in **nanoseconds** since the Unix epoch |
| `reader.duration` | `end_time - start_time`, also in nanoseconds |
| `reader.messages(connections=...)` | the iterator over `(connection, timestamp, rawdata)` |

`duration` is **not a field in the data**. It is a property the `rosbags` library computes from the bag's own index, which is why `reader.duration` works without anything being read first. It is the length of the recording, and because it is in nanoseconds every use of it below divides by `1e9`.

### One message, taken apart

The cell below opens `gnss.bag`, pulls the very first message on `/xsens/gnss`, and shows both what is stored and what comes out after decoding.

The things to notice: the payload really is just bytes until a schema is applied, and there are genuinely **two** timestamps. `header.frame_id` is also worth seeing, because it names the physical point on the vehicle that the measurement refers to.

In [2]:
with AnyReader([BAGS["gnss"]]) as reader:
    conn = next(c for c in reader.connections if c.topic == "/xsens/gnss")

    for connection, bag_timestamp, rawdata in reader.messages(connections=[conn]):
        print("STORED IN THE BAG")
        print(f"  topic          {connection.topic}")
        print(f"  message type   {connection.msgtype}")
        print(f"  bag timestamp  {bag_timestamp} ns   <- when the recorder wrote it")
        print(f"  payload        {len(rawdata)} bytes")
        print(f"                 {rawdata[:20].hex(' ')} ...")

        msg = reader.deserialize(rawdata, connection.msgtype)

        print("\nAFTER DECODING WITH THE SCHEMA")
        print(f"  header.stamp     {msg.header.stamp.sec}.{msg.header.stamp.nanosec:09d}"
              "   <- when the sensor says it measured")
        print(f"  header.frame_id  {msg.header.frame_id!r}")
        print(f"  latitude         {msg.latitude}")
        print(f"  longitude        {msg.longitude}")
        print(f"  altitude         {msg.altitude}")
        break

STORED IN THE BAG
  topic          /xsens/gnss
  message type   sensor_msgs/msg/NavSatFix
  bag timestamp  1698855377886621037 ns   <- when the recorder wrote it
  payload        130 bytes
                 d5 00 00 00 d1 79 42 65 37 9e d3 34 0e 00 00 00 78 73 65 6e ...

AFTER DECODING WITH THE SCHEMA
  header.stamp     1698855377.886283831   <- when the sensor says it measured
  header.frame_id  'xsens_imu_link'
  latitude         44.238514599999995
  longitude        -76.5004153
  altitude         72.849


### What a connection is

A **connection** is the bag's header record for one stream: "a publisher wrote messages of this type, on this topic". It holds the topic name, the message type, how many messages there are, and `msgdef`, which is the **full text of the schema** embedded in the file.

Two consequences, both of which this project depends on:

- **Listing connections requires no decoding.** That is why an inventory works even for message types we cannot yet interpret.
- **The bag describes itself.** The schema travels inside the file, so the custom NavINST types need nothing installed. Notebook 02 relies on this.

A topic can have more than one connection, for instance if a publisher restarted mid-recording, in which case its message count is split across them. The inventory below aggregates, so each topic appears once.

In [3]:
with AnyReader([BAGS["gnss"]]) as reader:
    print(f"{len(reader.connections)} connections in gnss.bag\n")
    for c in reader.connections:
        print(f"  id={c.id}  {c.topic:28s} {c.msgtype:36s} {c.msgcount:>7d} messages")

    conn = next(c for c in reader.connections if c.topic == "/xsens/gnss")
    schema = str(conn.msgdef.data)

print(f"\nThe embedded schema for {conn.msgtype}, first 16 lines:\n")
for line in schema.splitlines()[:16]:
    print("   ", line)

5 connections in gnss.bag

  id=0  /xsens/filter/quaternion     geometry_msgs/msg/QuaternionStamped   106500 messages
  id=1  /xsens/filter/twist          geometry_msgs/msg/TwistStamped        106500 messages
  id=2  /xsens/filter/positionlla    geometry_msgs/msg/Vector3Stamped      106500 messages
  id=3  /novatel/gps                 gps_common/msg/GPSFix                  53250 messages
  id=4  /xsens/gnss                  sensor_msgs/msg/NavSatFix               4260 messages

The embedded schema for sensor_msgs/msg/NavSatFix, first 16 lines:

    # Navigation Satellite fix for any Global Navigation Satellite System
    #
    # Specified using the WGS 84 reference ellipsoid
    
    # header.stamp specifies the ROS time for this measurement (the
    #        corresponding satellite time may be reported using the
    #        sensor_msgs/TimeReference message).
    #
    # header.frame_id is the frame of reference reported by the satellite
    #        receiver, usually the location of

### Building the inventory

One row per `(topic, message type)`, with:

- **`count`** the number of messages on that topic, summed over its connections.
- **`rate_hz`** that count divided by the recording duration, so 4 Hz means roughly 4 messages per second. It is an **average**, which cannot distinguish a steady stream from one with a long hole in it. Notebook 02 looks at the actual gaps.
- **`custom`** a flag computed here, not something in the data. It is `True` when the message type does not begin with a standard ROS package name, meaning the type was defined by NavINST rather than by ROS itself.

In [4]:
def inventory(bag_path: Path, label: str) -> pd.DataFrame:
    """Return one row per (topic, message type) with counts and rates."""
    with AnyReader([bag_path]) as reader:
        duration_s = reader.duration / 1e9
        rows = [
            {"topic": c.topic, "msgtype": c.msgtype, "count": c.msgcount}
            for c in reader.connections
        ]

    df = (
        pd.DataFrame(rows)
        .groupby(["topic", "msgtype"], as_index=False)["count"]
        .sum()  # collapse duplicate connections for the same topic
    )
    df.insert(0, "bag", label)
    df["rate_hz"] = (df["count"] / duration_s).round(1)
    df["custom"] = ~df["msgtype"].str.startswith(
        ("sensor_msgs/", "geometry_msgs/", "std_msgs/", "nav_msgs/")
    )
    df.attrs["duration_s"] = duration_s
    return df.sort_values("topic").reset_index(drop=True)


tables = {name: inventory(path, f"{name}.bag") for name, path in BAGS.items()}
both = pd.concat(tables.values(), ignore_index=True)

duration_s = tables["gnss"].attrs["duration_s"]
print(f"duration: {duration_s:.1f} s  ({duration_s / 60:.2f} min)")

duration: 1065.0 s  (17.75 min)


### `gnss.bag`

The bag that decides whether this project is viable. Look for a GNSS fix topic and check its rate against the 1 to 10 Hz the receiver spec allows.

Expect more than one plausible candidate here, and read the cell after the table before choosing between them.

In [5]:
tables["gnss"]

,bag,topic,msgtype,count,rate_hz,custom
0,gnss.bag,/novatel/gps,gps_common/msg/GPSFix,53250,50.0,True
1,gnss.bag,/xsens/filter/positionlla,geometry_msgs/msg/Vector3Stamped,106500,100.0,False
2,gnss.bag,/xsens/filter/quaternion,geometry_msgs/msg/QuaternionStamped,106500,100.0,False
3,gnss.bag,/xsens/filter/twist,geometry_msgs/msg/TwistStamped,106500,100.0,False
4,gnss.bag,/xsens/gnss,sensor_msgs/msg/NavSatFix,4260,4.0,False


### Two traps in `gnss.bag`

**`/novatel/gps` is ground truth hardware.** It is the NovAtel receiver, the other half of the reference system. It sits in the same bag as the input, at an attractive 50 Hz, and `gps_common/msg/GPSFix` even carries DOP fields. Using it as a filter input would be feeding the answer key into the filter.

**`/xsens/filter/*` is the Xsens's own fused solution.** The MTi-670G is not a bare IMU, it is a GNSS/INS device running its own internal estimator. The `positionlla`, `quaternion` and `twist` topics are the output of *that* filter: a position already fused from inertial and satellite data, at a tempting 100 Hz rather than the raw fix rate.

Using those would not be cheating against ground truth, but it would defeat the point of the project. The deliverable is a sensor fusion filter written from scratch. Taking a vendor's already-fused position as a measurement means the fusion has been done for you, by someone else, with an undisclosed algorithm, and no comparison between an EKF and a UKF downstream of it would mean much.

The honest inputs are the **raw GNSS fix** here and the **raw** `/xsens/imu` from `imus.bag`. This belongs in the README as a stated decision alongside the NovAtel split, because it is exactly the kind of choice a reviewer looks for.

### `imus.bag`

Expect the Xsens MTi-670G at 100 Hz and the OBD2 speed at roughly 16 Hz, plus several IMUs belonging to other sensors, since the Livox LiDAR and both ZED cameras carry their own. `/novatel/imu` is the tactical-grade reference IMU and is also off limits.

In [6]:
tables["imus"]

,bag,topic,msgtype,count,rate_hz,custom
0,imus.bag,/livox/imu,sensor_msgs/msg/Imu,215449,202.3,False
1,imus.bag,/novatel/imu,sensor_msgs/msg/Imu,213006,200.0,False
2,imus.bag,/obd2/maf,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
3,imus.bag,/obd2/rpm,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
4,imus.bag,/obd2/speed,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
5,imus.bag,/xsens/imu,sensor_msgs/msg/Imu,106500,100.0,False
6,imus.bag,/xsens/mag,geometry_msgs/msg/Vector3Stamped,106500,100.0,False
7,imus.bag,/xsens/pressure,sensor_msgs/msg/FluidPressure,106500,100.0,False
8,imus.bag,/xsens/temperature,sensor_msgs/msg/Temperature,106500,100.0,False
9,imus.bag,/zedx_left/imu,sensor_msgs/msg/Imu,106432,99.9,False


### `reference.bag`

Ground truth only. For urban sequences this is the post-processed NovAtel GNSS/INS solution, PPK corrected to roughly 1 cm for Urban01 to Urban04. Unlike the indoor LiDAR map reference, it does **not** fuse the OBD odometer, so it shares nothing with the filter inputs.

In [7]:
tables["reference"]

,bag,topic,msgtype,count,rate_hz,custom
0,reference.bag,/novatel/reference/postprocessed/inspva,navinst_ros/msg/INSPVA,53251,50.0,True
1,reference.bag,/novatel/reference/postprocessed/inspvax,navinst_ros/msg/INSPVAX,1066,1.0,True


## Roles: what is an input, and what is the answer key

This is the distinction the whole project rests on, so it is worth stating precisely.

### The sensors

| Role | Device | What it physically is | Rate |
|---|---|---|---|
| **Input** | Xsens MTi-670G IMU | three MEMS gyroscopes measuring angular rate, three accelerometers measuring specific force | 100 Hz |
| **Input** | Xsens GNSS, a u-blox ZED-F9 inside the same unit | satellite receiver producing an absolute position fix | 4 Hz |
| **Input** | OBD-II scanner on the car's diagnostic port | reads forward speed from the vehicle's own wheel sensors | 16.7 Hz |
| **Ground truth only** | NovAtel PwrPak7-E1 with a KVH1750 IMU, post-processed | survey-grade receiver plus tactical-grade IMU | 50 Hz |

"Xsens" is the manufacturer. The MTi-670G is one unit containing an IMU, a magnetometer, a pressure sensor, a GNSS receiver **and a processor running Xsens's own fusion algorithm**. That last component is the reason for the exclusions above.

### Why the NovAtel solution counts as truth

It is not simply a better sensor. Four things separate it from the input:

1. **Better receiver**, with a survey-grade antenna, rather than a consumer chip.
2. **A tactical-grade IMU beside it.** The KVH1750 drifts at 0.05°/h against the Xsens's 8°/h, roughly 160 times less.
3. **Differential correction against stationary base stations.** A second receiver at a surveyed, known position recorded the same satellites at the same time, so the atmospheric and orbital error it saw could be subtracted from the moving one.
4. **Post-processing, which is the real reason.** The solution was computed afterwards in NovAtel Inertial Explorer, so it **can look forwards in time**: if the position at $t$ is uncertain, the processing already knows what happened at $t+1$ and uses it. A filter running causally never can.

The result is about 1 cm, against roughly 0.5 m to several metres for the Xsens fix. That gap is what makes it fair to treat the reference as exact while grading an estimate that errs by metres.

**In one sentence:** the reference comes from better hardware, corrected against surveyed base stations, and smoothed offline using data from the future, none of which a real-time filter on a low-cost sensor can do.

### Where each input enters the filter

A point that is easy to get backwards, and expensive when you do:

| Role | Signal | Enters as |
|---|---|---|
| **Input** $u$ | IMU angular rate and acceleration | the **prediction** step, $\hat{x} \leftarrow f(\hat{x}, u)$ |
| **Measurement** $y$ | GNSS position | an **update** |
| **Measurement** $y$ | OBD forward speed | an **update** |
| **Measurement** $y$ | non-holonomic constraint | an **update** |

**The IMU is not a measurement.** It drives the prediction, and it never appears in $y$. The consequence is that IMU noise enters through $Q$, the process noise, and not through $R$. All of `CLAUDE.md` §6's conversion of datasheet noise densities into process noise is exactly this step. See `Kalman filter.md` section 7.

### Everything else

The Livox and ZED IMUs, pressure, temperature, magnetometer, RPM and MAF are all out of scope. The magnetometer is excluded deliberately, per §3: an urban canyon is full of steel and reinforced concrete, which distorts the local field and invalidates hard-iron calibration, so it would inject a confident wrong heading in exactly the places where GNSS is already struggling.

In [8]:
# The streams that matter, and nothing else.
keep = ["/xsens/gnss", "/xsens/imu", "/obd2/speed"]

selected = both[
    both["topic"].isin(keep) | both["topic"].str.contains("reference", case=False)
]
selected[["bag", "topic", "msgtype", "count", "rate_hz", "custom"]]

,bag,topic,msgtype,count,rate_hz,custom
4,gnss.bag,/xsens/gnss,sensor_msgs/msg/NavSatFix,4260,4.0,False
9,imus.bag,/obd2/speed,navinst_obd_driver/msg/ObdMeasurmentStamped,17751,16.7,True
10,imus.bag,/xsens/imu,sensor_msgs/msg/Imu,106500,100.0,False
18,reference.bag,/novatel/reference/postprocessed/inspva,navinst_ros/msg/INSPVA,53251,50.0,True
19,reference.bag,/novatel/reference/postprocessed/inspvax,navinst_ros/msg/INSPVAX,1066,1.0,True


In [9]:
# Every custom type present across the three bags. These need registering with
# rosbags before any of their messages can be decoded.
sorted(both.loc[both["custom"], "msgtype"].unique())

['gps_common/msg/GPSFix',
 'navinst_obd_driver/msg/ObdMeasurmentStamped',
 'navinst_ros/msg/INSPVA',
 'navinst_ros/msg/INSPVAX']

## Notes and gotchas

**The package name in the bag may not match the cloned repository.** The OBD type is declared as `navinst_obd_driver/msg/ObdMeasurmentStamped`, while the definition in `~/repos/dataset` lives under `navinst_ros/msg/`. When registering types manually, the name must match what the *bag* says, or lookup fails. The cleanest way to sidestep this entirely is to register from each connection's own embedded `.msgdef`, since ROS1 bags carry the full definition text internally and so can describe themselves.

**Two timestamps exist per message:** the bag's record time, which is what `reader.messages()` yields, and the sensor's own `header.stamp` inside the message. They differ by driver and buffering latency, which is not constant. **Always use `header.stamp`** for time alignment. Fusing a 100 Hz IMU with a few-hertz GNSS at urban driving speeds means tens of milliseconds of timing error turn straight into position error.

**Message count divided by duration is an average**, not proof of a steady rate. A sensor that dropped out for thirty seconds and ran fast elsewhere averages out to something plausible. For GNSS that distinction is the whole project: the gaps between consecutive fixes, and the stretches where quality collapses, are what E3 measures.

---

### Next

1. Register the custom types and decode one message of each topic of interest. In particular, inspect the GNSS message's quality fields (`status`, `position_covariance`, `position_covariance_type`). That decides which branch of §14's adaptive-$R$ table applies.
2. Plot the GNSS track against the reference and find where the fix degrades. How many complete dropouts are there, and how long are they?
3. Then write `src/data_loader.py` properly.